# 10.2 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/10-deployment-mlops/02-fastapi/solutions.ipynb)

In [1]:
# --- course files: found locally, downloaded on Colab -------------------------
import sys, urllib.request
from pathlib import Path
REPO_RAW = "https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/"

def repo_file(rel):
    """Return a local path to a file in this repo, downloading it if needed."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / rel).exists():
            return parent / rel
    dest = Path("course_files") / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
repo_file("data/download.py")                                  # churn_model finds the data through it
sys.path.insert(0, str(repo_file("lib/churn_model.py").parent))
import churn_model as cm

bundle = cm.load_bundle()          # trains and saves models/churn_bundle.joblib the first time
ROOT = cm.repo_root()
print(f"Model trained {bundle['trained']}, test AUC {bundle['metrics']['test_auc']}, "
      f"threshold {bundle['threshold']}")

Model trained 2026-10-28, test AUC 0.741, threshold 0.07


In [2]:
import importlib.util

def load_module(rel, name):
    """Import a .py file from the course as a module (so the notebook can test it)."""
    spec = importlib.util.spec_from_file_location(name, repo_file(rel))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

from fastapi.testclient import TestClient
api = load_module("modules/10-deployment-mlops/02-fastapi/api.py", "api")
client = TestClient(api.app)            # sends requests straight to the app: no server, no port

## Exercise 1

Sorted: 2, 2, 2, 2, 2, 3, 3, 3, 4, 15.

- p50: between the 5th and 6th values, 2 and 3 → **2.5 ms** with NumPy's interpolation (a "middle value" answer of 2 or 3 is fine on paper).
- p95: the 10th value on paper, **15 ms**; NumPy interpolates between 4 and 15.
- mean = 38 / 10 = **3.8 ms**; throughput ≈ 1 / 0.0038 ≈ **263 requests per second**.

In [3]:
t = np.array([3, 2, 2, 4, 2, 3, 15, 2, 3, 2])
print(f"p50 {np.percentile(t, 50)}  p95 {np.percentile(t, 95):.2f}  mean {t.mean()}  "
      f"throughput {1000 / t.mean():.0f}/s")

p50 2.5  p95 10.05  mean 3.8  throughput 263/s


## Exercise 2

In [4]:
@api.app.get("/model")
def model_info():
    """What data the model was trained on, and how well it scored."""
    return {"data": api.bundle["data"], "metrics": api.bundle["metrics"],
            "trained": api.bundle["trained"]}

r = client.get("/model")
print(r.status_code, r.json())

200 {'data': '3,317 customers, orders before 2011-09-01, churn = no order in next 91 days', 'metrics': {'cv_auc': 0.738, 'test_auc': 0.741}, 'trained': '2026-10-28'}


## Exercise 3

In [5]:
from sklearn.model_selection import train_test_split
cust = cm.build_customer_table(cm.load_sales())
X_train, X_test, y_train, y_test = train_test_split(
    cust[cm.FEATURES], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"{len(cust):,} customers, train {len(X_train):,} / test {len(X_test):,}")

rows = X_test.head(50).astype({"spend": float}).to_dict(orient="records")
rows[10]["orders"] = 0
r = client.post("/predict/batch", json=rows)
print(r.status_code, [(e["loc"], e["msg"]) for e in r.json()["detail"]])

3,317 customers, train 2,487 / test 830
422 [(['body', 10, 'orders'], 'Input should be greater than or equal to 1')]


The whole batch is rejected with 422, and the error says exactly which row and field (`body`, row 10, `orders`). That is safe, but for a nightly batch of 40,000 customers one bad row would stop everything.

An alternative: accept a list of loosely typed rows, validate each one inside the endpoint, and return `{"predictions": [...], "rejected": [{"row": 10, "error": "..."}]}`. The caller gets every valid score plus a clear list of rows to fix. That is what `score_customers.py` in 10.1 does for CSV files.